  %md
   WanderBrick : A Day as a Data Analyst
   Mission 2: Turn Data Into Answers
 
  **The 3 PM request (from the Product Owner):**
  1. Total booking revenue
  2. Top destination cities
  3. Booking status mix (confirmed / completed / cancelled / pending)
  4. Monthly booking trend
 
  The Data Engineer has already published the **Gold** materialized view
  workspace.`03_gold`.gold_bookings (Bronze -> Silver -> Gold pipeline).
 
  > **Key point about this Gold table:** it is **pre-aggregated to one row per
  > check_in_date + city.** It is *not* one row per booking. Status is already
  > rolled up into count columns (confirmed_bookings, cancelled_bookings, ...),
  > and revenue/nights/guests are already summed per day+city.
  >
  > So the analyst **sums the pre-aggregated columns and groups by city/month** :
  > there is no WHERE status = 'confirmed' filter here.
 
  No setup cell needed : we read straight from the engineer's Gold view.

In [0]:


GOLD = "workspace.`03_gold`.gold_bookings"
print("Reading from:", GOLD)
spark.sql(f"SELECT * FROM {GOLD} LIMIT 5").display()



### DEMO STARTS HERE
### Step 1 : Preview the trusted Gold view
#####"This is governed, business-ready data, already aggregated per day and city.


In [0]:
%sql

SELECT check_in_date, city, total_bookings, confirmed_bookings,
cancelled_bookings, pending_bookings, total_revenue, avg_nights
FROM workspace.`03_gold`.gold_bookings
ORDER  BY check_in_date
LIMIT  10;



  %md
   Step 2 : Q1: Total revenue & booking funnel
  Because the grain is day+city, we **SUM** the pre-aggregated measures across all rows.

In [0]:
  %sql
  SELECT
    SUM(total_bookings)                                   AS total_bookings,
    SUM(confirmed_bookings)                               AS confirmed,
    SUM(completed_bookings)                               AS completed,
    SUM(cancelled_bookings)                               AS cancelled,
    SUM(pending_bookings)                                 AS pending,
    ROUND(SUM(total_revenue), 2)                          AS total_revenue,
    ROUND(SUM(total_revenue) / SUM(total_bookings), 2)    AS avg_revenue_per_booking
  FROM workspace.`03_gold`.gold_bookings;

  %md
  **Expected (from the demo dataset):** ~47 city-days, **~$27,388** total revenue,
  and a funnel of ~17 confirmed / 4 completed / 13 cancelled / 18 pending.
  Turn total_revenue into a **Counter** tile; the funnel into a small **bar**.
 
  Talking point: only ~45% of bookings are confirmed+completed : a natural
  "why do so many stay pending or cancel?" question for the business.

  %md
   Step 3 : Q2: Top destination cities  (headline chart)
  After running, click **+ -> Visualization -> Bar chart** (X = city, Y = revenue).

In [0]:
  %sql
  SELECT city,
         ROUND(SUM(total_revenue), 2)  AS revenue,
         SUM(total_bookings)           AS bookings,
         SUM(confirmed_bookings)       AS confirmed
  FROM   workspace.`03_gold`.gold_bookings
  GROUP  BY city
  ORDER  BY revenue DESC;



  %md
  **Expected:** **Bali (~$6,936)** and **Lisbon (~$6,786)** lead, then Reykjavik and Cape Town.
 
  Because this is the Gold view, Tokyo/Tokio are already merged into one **Tokyo** row:
  that de-duplication happened in **Silver (Mission 3)**. Great moment to say
  *"the analyst can trust this because the engineer already fixed it upstream."*

  %md
   Step 4 :  Q3: Booking status mix by city (the funnel view)
  Shows healthy vs. leaky demand per city. Good as a **stacked bar**.


In [0]:
  %sql
  SELECT city,
         SUM(confirmed_bookings)  AS confirmed,
         SUM(completed_bookings)  AS completed,
         SUM(pending_bookings)    AS pending,
         SUM(cancelled_bookings)  AS cancelled
  FROM   workspace.`03_gold`.gold_bookings
  GROUP  BY city
  ORDER  BY confirmed DESC;

  %md
   Step 5 : Q4: Monthly booking trend
  check_in_date is already a DATE in Gold, so we just format to month -> **line chart**.

In [0]:
  %sql
  SELECT date_format(check_in_date, 'yyyy-MM')  AS month,
         ROUND(SUM(total_revenue), 2)           AS revenue,
         SUM(total_bookings)                    AS bookings
  FROM   workspace.`03_gold`.gold_bookings
  GROUP  BY 1
  ORDER  BY 1;

  %md
  **Expected:** a strong summer peak : **June 2025 (~$7,527)** and **July 2025 (~$6,625)**
  carry most revenue.
 
  **Data-quality catch (do this live):** you'll also see an **Aug-2024 month with
  *negative* revenue** and a **$0 month**. Ask the room: *"Can a month really have negative
  revenue?"* -> It traces to bad/null amounts in the raw data. On a dashboard you'd filter
  these out  and flag them back to the Data Engineer. A clean bridge to Mission 3.

  %md
   Step 6 (bonus) : Operational metrics the Gold view unlocks for free
  Because the engineer pre-computed nights, guests and property counts, the analyst gets
  richer KPIs with no extra work.

In [0]:
  %sql
  SELECT city,
         SUM(total_nights)                         AS total_nights,
         ROUND(AVG(avg_nights), 2)                 AS avg_nights_per_booking,
         SUM(total_guests)                         AS total_guests,
         SUM(distinct_properties)                  AS property_days,
         ROUND(AVG(avg_price_per_night), 2)        AS avg_price_per_night
  FROM   workspace.`03_gold`.gold_bookings
  GROUP  BY city
  ORDER  BY total_nights DESC;

  %md
  ---
   Step 7 :  Build the dashboard
  1. On each result above, click **+ -> Add to dashboard** (or save as a visualization).
  2. Assemble: a **Counter** (total revenue), a **bar** (revenue by city),
     a **stacked bar** (status mix), and a **line** (monthly trend).
  3. **Share** the dashboard link with the Product Owner.
 
  *"No spreadsheet emailed. No CSV exported. The business opens one governed link."*

  %md
   Step 8 :  The finale: AI/BI Genie
  Open a **Genie space** on workspace.`03_gold`.gold_bookings and ask in plain English:
 
  > *"What was our total revenue, and which city earned the most?"*
  >
  > *"Show revenue by month as a line chart."*
  >
  > *"Which city has the highest cancellation count?"*
 
  Same governed view, **no SQL required** — a natural bridge to **Mission 5: GenAI.**

   Recap :  the analyst's path
  **Business Question -> Gold view -> SQL Warehouse -> SQL -> Dashboard -> Business User**
 
  The Gold view is **pre-aggregated per day + city**, so the analyst *sums and groups* rather
  than filtering raw rows. Four business questions answered end-to-end, without leaving the
  platform  and the quirks we spotted (negative-revenue month, high pending rate) hand
  straight back to the **Data Engineer (Mission 3).**